# [title] One-state Model: a posterior network trained with BayesFlow

**Question.** Can a neural network, trained once on simulated Sittings of the one-state
Model, give a trustworthy posterior of the Model's four parameters for a new Sitting, at
full length and without downsampling?

**Scope.** Step C3b of [issue #12](https://github.com/opherdonchin/MotorAdpatationSBI/issues/12).
Background: the wiki pages
[Concept-amortized-inference](https://github.com/opherdonchin/MotorAdpatationSBI/wiki/Concept-amortized-inference)
(what is trained, and how it is checked) and
[Concept-summary-networks](https://github.com/opherdonchin/MotorAdpatationSBI/wiki/Concept-summary-networks)
(how the network reads a Sitting). Why this summary network:
[decision 0007](../../docs/decisions/0007-summary-network-for-sittings.md). The
Model, Prior and Schedule Design: [docs/models/one_state.md](../../docs/models/one_state.md).
Terms: [glossary](../../docs/glossary.md).

**What is built.** Three Simulated Experiments (training, validation, test) with the Prior
and Schedule Design of decision 0005, and an Inference Engine: a summary network
(convolution, then recurrence) that reads every Trial of a Sitting ($p_t$, $v_t$, $y_t$), followed by a
flow-matching inference network that turns its 12 summary numbers into draws from the
posterior of $A$, $B$, $\sigma_\eta$ and $\sigma_\varepsilon$.

**Checks:**

1. *Training converged:* the loss on the validation Experiment has flattened and has not
   pulled away from the training loss.
2. *Calibration* (the Check that decides whether to trust the Engine): over the test
   Sittings, the true value of each parameter falls in the posterior's intervals as
   often as it should.
3. *Recovery and contraction* (reported, not pass or fail): how close the posterior's
   centre is to the true value, and how much narrower the posterior is than the Prior.
4. *Sitting length:* Checks 2 and 3 separately for the shorter and the longer half of
   the test Sittings.
5. *Padding* (reported): how much the summary of a Sitting changes when it is padded to
   a length other than the one used in training.

In [ ]:
# [imports]
import json
import pathlib
import subprocess

import bayesflow as bf
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import special, stats

from designs import visuomotor_adaptation_experiment as des
from models import one_state as mdl
from motor_sbi import experiments

In [ ]:
# [config] Root seed, provenance, Prior and Schedule Design, sizes, networks, training
root_seed = 112907950985685676175328411057333210321  # secrets.randbits(128)

# Provenance: the repository's top folder, the commit that produced the outputs (with
# -dirty if there were uncommitted changes), and where this notebook's outputs go.
REPO = pathlib.Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
GIT_COMMIT = subprocess.check_output(["git", "describe", "--always", "--dirty"], text=True).strip()
DATA_DIR = REPO / "data" / "simulated"
OUT_DIR = REPO / "outputs" / "one_state" / "engines" / "posterior_random_blocks"

# The Prior and Schedule Design of decision 0005, from their stated 95% ranges.
z95 = stats.norm.ppf(0.975)
range_a, range_b = (0.75, 0.999), (0.01, 0.5)
range_sigma_epsilon, range_ratio, range_block_len = (1 / 15, 1 / 3), (1 / 15, 1 / 5), (10, 250)
prior = {
    "mu_logit_A": special.logit(range_a).mean(), "sigma_logit_A": np.ptp(special.logit(range_a)) / (2 * z95),
    "mu_logit_B": special.logit(range_b).mean(), "sigma_logit_B": np.ptp(special.logit(range_b)) / (2 * z95),
    "mu_log_sigma_epsilon": np.log(range_sigma_epsilon).mean(),
    "sigma_log_sigma_epsilon": np.ptp(np.log(range_sigma_epsilon)) / (2 * z95),
    "mu_log_ratio": np.log(range_ratio).mean(), "sigma_log_ratio": np.ptp(np.log(range_ratio)) / (2 * z95),
}
schedule_design = {
    "n_blocks_min": 5, "n_blocks_max": 16,
    "mu_log_block_len": np.log(range_block_len).mean(),
    "sigma_log_block_len": np.ptp(np.log(range_block_len)) / (2 * z95),
    "trial_type_concentration": 3.0,
}

# One Simulated Experiment per purpose, each in data/simulated/one_state_random_blocks_<purpose>.
# 20,000 training Sittings and 300 for validation are BayesFlow's guidance for a first pass
# with a fast Simulator; 300 test Sittings as agreed for the Checks.
n_sittings = {"train": 20_000, "validation": 300, "test": 300}
folders = {purpose: DATA_DIR / f"one_state_random_blocks_{purpose}" for purpose in n_sittings}

# Networks at BayesFlow's "Base" size, the size its guidance says to start with.
summary_dim = 12  # 3 summary numbers per parameter (BayesFlow's starting heuristic)
conv_filters = (32, 64)  # two convolution layers, each over a window of conv_window Trials
conv_window = 3
recurrent_units = 128  # size of the state the recurrent layers carry from Trial to Trial
flow_widths = (256, 256, 256, 256)
flow_time_embedding_dim = 32

epochs = 100
batch_size = 32
n_posterior_draws = 1000  # per test Sitting, for the Checks (BayesFlow's default)

In [ ]:
# [rng] One stream per Simulated Experiment, and a seed for Keras from the root stream
rng = np.random.default_rng(root_seed)
rng_experiment = dict(zip(n_sittings, rng.spawn(len(n_sittings))))
keras.utils.set_random_seed(int(rng.integers(2**31)))

## [simulate] The three Simulated Experiments

Each Sitting has its own parameter values (from the Prior) and its own Schedule (from the
Schedule Design). The three Experiments use separate random streams and are saved like
any other Experiment, with their Ground Truth apart.

In [ ]:
# [simulate-and-save] Simulate and save the training, validation and test Experiments
for purpose, n in n_sittings.items():
    sittings, ground_truth = experiments.simulate_experiment(
        rng_experiment[purpose], mdl, prior, des.sample_schedule, schedule_design, n
    )
    record = {
        "model": mdl.__name__,
        "schedule_generator": f"{des.__name__}.sample_schedule",
        "prior": prior,
        "schedule_design": schedule_design,
        "n_sittings": n,
        "purpose": purpose,
        "root_seed": str(root_seed),
        "git_commit": GIT_COMMIT,
    }
    experiments.save_experiment(folders[purpose], sittings, ground_truth, record)
del sittings, ground_truth

## [arrays] From Sittings to the arrays BayesFlow trains on

A network works on batches of Sittings in one array, so all Sittings must have one
length. Shorter Sittings are **padded** with zeros at the end, up to the longest Sitting
of the three Experiments. The summary network we use cannot be told to skip padded
Trials, so every Trial carries a fourth number, `real`: 1 on a real Trial and 0 on padding
(where $p$, $v$ and $y$ are 0 too). Nothing is dropped and nothing is downsampled: every
Trial of every Sitting is in the array.

The parameter values of the Ground Truth are attached here because training needs them
(they are what the network learns to predict) and the Checks need them. An Analysis of an
Actual Experiment would have only the Sittings.

In [ ]:
# [pad] Per Experiment: p, v, y and the real-Trial flag as arrays (Sittings, Trials); parameters (Sittings, 1)
loaded = {purpose: experiments.load_sittings(folder) for purpose, folder in folders.items()}
lengths = {purpose: np.array([len(s["y"]) for s in sittings]) for purpose, sittings in loaded.items()}
max_trials = max(n.max() for n in lengths.values())
series_vars = mdl.TASK.CONDITION_VARS + mdl.TASK.OBSERVATION_VARS

data = {}
for purpose, sittings in loaded.items():
    arrays = {name: np.zeros((len(sittings), max_trials), dtype="float32") for name in series_vars}
    arrays["real"] = (np.arange(max_trials) < lengths[purpose][:, None]).astype("float32")
    for i, sitting in enumerate(sittings):
        for name in series_vars:
            arrays[name][i, : lengths[purpose][i]] = sitting[name]
    truth, _ = experiments.load_ground_truth(folders[purpose])
    for name in mdl.PARAMETERS:
        arrays[name] = truth[name][:, None].astype("float32")
    data[purpose] = arrays
del loaded

pd.DataFrame({
    purpose: {
        "Sittings": len(n), "shortest": n.min(), "median": int(np.median(n)), "longest": n.max(),
        "padded to": max_trials, "share of the array that is padding": round(1 - n.mean() / max_trials, 2),
    }
    for purpose, n in lengths.items()
})

## [engine] The Inference Engine

**The adapter** says which array goes where. `p`, `v`, `y` and `real` together are what
the summary network reads, one row of four numbers per Trial. The four parameters are what the inference network learns
the distribution of; `constrain` moves them to unbounded scales first ($A$ and $B$ by the
logit, the two noise sizes by a log-like transform), so the network can never propose a
value outside a parameter's domain.

**The summary network** is BayesFlow's `TimeSeriesNetwork`: two convolution layers compute
features of each Trial and its neighbours (5 Trials in all), and recurrent layers then
read those features Trial by Trial in both directions, carrying a state of 128 numbers.
It reads padded Trials like any others and has to learn from `real` to let them pass.
**The inference network** is flow matching, BayesFlow's
recommended default.

Only the parameters are standardized by BayesFlow. The Sittings are already in units of
the perturbation size, and standardizing them would mix the padding into the averages.

In [ ]:
# [adapter] Route the arrays: p, v, y and real to the summary network; the parameters
adapter = (
    bf.Adapter()
    .as_time_series([*series_vars, "real"])
    .constrain(["A", "B"], lower=0, upper=1)
    .constrain(["sigma_eta", "sigma_epsilon"], lower=0)
    .concatenate([*series_vars, "real"], into="summary_variables")
    .concatenate(list(mdl.PARAMETERS), into="inference_variables")
)

In [ ]:
# [workflow] The two networks and the BayesFlow workflow that trains and saves them
OUT_DIR.mkdir(parents=True, exist_ok=True)
workflow = bf.BasicWorkflow(
    adapter=adapter,
    summary_network=bf.networks.TimeSeriesNetwork(
        summary_dim=summary_dim, filters=conv_filters, kernel_sizes=conv_window, recurrent_dim=recurrent_units
    ),
    inference_network=bf.networks.FlowMatching(
        subnet_kwargs={"widths": flow_widths, "time_embedding_dim": flow_time_embedding_dim}
    ),
    standardize="inference_variables",
    checkpoint_filepath=str(OUT_DIR),
    checkpoint_name="engine",
)

## [train] Training

Each epoch shows the network every training Sitting once, in batches of 32. After each
epoch the Engine is saved to `engine.keras` and the loss is computed on the validation
Experiment, which the network never trains on.

In [ ]:
# [fit] Train, then save the loss history and the record of what produced the Engine
history = workflow.fit_offline(
    data=data["train"], epochs=epochs, batch_size=batch_size, validation_data=data["validation"], verbose=2
)
_ = (OUT_DIR / "history.json").write_text(json.dumps(history.history))
_ = (OUT_DIR / "record.json").write_text(json.dumps({
    "training_experiment": folders["train"].name,
    "validation_experiment": folders["validation"].name,
    "summary_network": "TimeSeriesNetwork", "summary_dim": summary_dim, "conv_filters": conv_filters,
    "conv_window": conv_window, "recurrent_units": recurrent_units, "padded_to": int(max_trials),
    "inference_network": "FlowMatching", "flow_widths": flow_widths,
    "epochs": epochs, "batch_size": batch_size,
    "root_seed": str(root_seed), "git_commit": GIT_COMMIT,
}, indent=2))

## [check-1] Check 1: training converged

In [ ]:
# [check-training] Training and validation loss per epoch
loss = pd.DataFrame(history.history)[["loss", "val_loss"]]
fig = bf.diagnostics.plots.loss(history)
last = loss.tail(10).mean()
pd.Series({
    "training loss, mean of last 10 epochs": round(last["loss"], 3),
    "validation loss, mean of last 10 epochs": round(last["val_loss"], 3),
    "validation loss, change over the last 20 epochs": round(
        loss["val_loss"].tail(10).mean() - loss["val_loss"].iloc[-20:-10].mean(), 3
    ),
    "epoch with the lowest validation loss": int(loss["val_loss"].idxmin()) + 1,
})

## [check-2] Checks 2 and 3: calibration, recovery and contraction on the test Experiment

For each test Sitting the Engine draws from its posterior; the same draws serve all the
Checks. The figures and the table are BayesFlow's own diagnostics.

- **Calibration** (`calibration_ecdf`, and *Calibration Error* in the table): for each
  parameter, where the true value ranks among the posterior draws. If the posterior is
  honest, the ranks are uniform and the curve stays inside the grey band.
- **Recovery** (`recovery`, and *NRMSE*): posterior median against true value.
- **Contraction** (`z_score_contraction`, and *Posterior Contraction*): 1 minus the
  posterior's variance over the Prior's; 0 means the Sitting taught nothing about the
  parameter, 1 means it pinned it down.

In [ ]:
# [sample-test] Posterior draws for every test Sitting
test = data["test"]
draws = workflow.sample(conditions=test, num_samples=n_posterior_draws)

In [ ]:
# [check-diagnostics] BayesFlow's default diagnostics: figures and table, both saved
figures = workflow.plot_default_diagnostics(test_data=test, samples=draws)
for name, fig in figures.items():
    fig.savefig(OUT_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
metrics = workflow.compute_default_diagnostics(test_data=test, samples=draws, as_data_frame=True)
metrics.to_csv(OUT_DIR / "metrics.csv")
metrics.round(3)

## [check-4] Check 4: does Sitting length matter?

The same table for the shorter and the longer half of the test Sittings. Longer Sittings
hold more information, so better recovery and more contraction are expected there. What
would be a warning is *calibration* that differs between the halves: it would mean the
Engine handles some lengths badly.

In [ ]:
# [check-length] Default diagnostics for the shorter and the longer half of the test Sittings
is_long = lengths["test"] >= np.median(lengths["test"])
halves = {"shorter half": ~is_long, "longer half": is_long}
by_length = pd.concat({
    f"{label} ({lengths['test'][rows].min()} to {lengths['test'][rows].max()} Trials)":
        workflow.compute_default_diagnostics(
            test_data={name: values[rows] for name, values in test.items()},
            samples={name: values[rows] for name, values in draws.items()},
            as_data_frame=True,
        )
    for label, rows in halves.items()
})
by_length.round(3)

## [check-5] Check 5: how much does padding matter?

In training, every Sitting was padded to the same total length, so that is how a Sitting
must be given to this Engine: padded to `max_trials`, and no longer than that. This Check
measures what happens otherwise. For a few test Sittings it compares the 12 summary
numbers when the Sitting is padded as in training, padded with 250 more Trials, and not
padded at all. The changes are to be read against the spread of the summary numbers
across Sittings: a change that is small beside that spread cannot move the posterior
much.

In [ ]:
# [check-padding] Summaries with the padding as trained, 250 Trials longer, and removed
n_checked = 8
extra_trials = 250
few = {name: values[:n_checked] for name, values in test.items()}
series = np.asarray(workflow.adapter(few)["summary_variables"])
summary_network = workflow.approximator.summary_network
as_trained = np.asarray(summary_network(series, training=False))

longer = np.concatenate([series, np.zeros((n_checked, extra_trials, series.shape[-1]), dtype=series.dtype)], axis=1)
with_more = np.asarray(summary_network(longer, training=False))

alone = np.concatenate([
    np.asarray(summary_network(series[i : i + 1, : lengths["test"][i]], training=False))
    for i in range(n_checked)
])
pd.Series({
    "spread of the summary numbers across Sittings (sd)": as_trained.std(axis=0).mean(),
    "largest change with 250 more padded Trials": np.abs(with_more - as_trained).max(),
    "largest change with the padding removed": np.abs(alone - as_trained).max(),
})

## [results] What this shows

*To be written from the executed notebook.*